In [0]:
# Updated hr_bronze.py
# Changes: Updated pathGlobFilter to "*.csv" to allow streaming ingestion of any new CSV files added to the directory.
# Assumed that new files follow the same schema and are added as additional CSV files (e.g., HR_Analytics_2025-11-17.csv).
# Also, made ROOT_PATH more generic; adjust as per your actual landing zone.

# Databricks notebook source

import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# -------------------------------
# Configuration
# -------------------------------
ROOT_PATH = "/Volumes/hrcatalog/landing/operational_data/IshitaMandlik2404/HRDemo/refs/heads/main/"  # Adjust to your actual landing directory for HR CSVs

# -------------------------------
# Schema Definition (Bronze)
# -------------------------------
hr_schema = StructType([
    StructField("EmpID", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("AgeGroup", StringType(), True),
    StructField("Attrition", StringType(), True),
    StructField("BusinessTravel", StringType(), True),
    StructField("DailyRate", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("DistanceFromHome", IntegerType(), True),
    StructField("Education", IntegerType(), True),
    StructField("EducationField", StringType(), True),
    StructField("EmployeeCount", IntegerType(), True),
    StructField("EmployeeNumber", IntegerType(), True),
    StructField("EnvironmentSatisfaction", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("HourlyRate", IntegerType(), True),
    StructField("JobInvolvement", IntegerType(), True),
    StructField("JobLevel", IntegerType(), True),
    StructField("JobRole", StringType(), True),
    StructField("JobSatisfaction", IntegerType(), True),
    StructField("MaritalStatus", StringType(), True),
    StructField("MonthlyIncome", IntegerType(), True),
    StructField("SalarySlab", StringType(), True),
    StructField("MonthlyRate", IntegerType(), True),
    StructField("NumCompaniesWorked", IntegerType(), True),
    StructField("Over18", StringType(), True),
    StructField("OverTime", StringType(), True),
    StructField("PercentSalaryHike", IntegerType(), True),
    StructField("PerformanceRating", IntegerType(), True),
    StructField("RelationshipSatisfaction", IntegerType(), True),
    StructField("StandardHours", IntegerType(), True),
    StructField("StockOptionLevel", IntegerType(), True),
    StructField("TotalWorkingYears", IntegerType(), True),
    StructField("TrainingTimesLastYear", IntegerType(), True),
    StructField("WorkLifeBalance", IntegerType(), True),
    StructField("YearsAtCompany", IntegerType(), True),
    StructField("YearsInCurrentRole", IntegerType(), True),
    StructField("YearsSinceLastPromotion", IntegerType(), True),
    StructField("YearsWithCurrManager", IntegerType(), True)
])

# -------------------------------
# BRONZE TABLE: Raw Ingestion
# -------------------------------
@dlt.table(
    name="bronze_hr_analytics",
    comment="Raw HR analytics data ingested from CSV (bronze layer)",
    table_properties={"quality": "bronze"}
)
def bronze_hr_analytics():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "false")
            .option("cloudFiles.schemaLocation", "/tmp/dlt_schema/hr_analytics")
            .option("header", "true")
            .option("delimiter", ",")
            .option("quote", "\"")
            .option("escape", "\"")
            .option("badRecordsPath", "/tmp/dlt_bad/hr_analytics")
            .option("pathGlobFilter", "*.csv")  # Updated to pick up any new CSV files added
            .schema(hr_schema)
            .load(ROOT_PATH)
            .withColumn("input_file_path", F.col("_metadata.file_path"))
            .withColumn("input_file_name", F.col("_metadata.file_name"))
            .withColumn("ingest_timestamp", F.current_timestamp())
    )
     